## 가계부(입출금 내역)앱의 데이터 처리 부분 단계 
최종 목표는 세 가지

1. 사용자가 입력하거나 CSV로 올린 수입·지출 데이터를 검증하기
2. 월별 통계를 계산하기
3. 결과를 CSV로 저장하기


## 1. 라이브러리와 상수(constant)
* 프로그램 전체에서 공통으로 사용할 컬럼 이름과 데이터 입력 제한 기준을 미리 정해둠
    * COLUMNS   / 날짜, 구분, 항목, 금액  / 필수 컬럼 이름 정의
    * MAX_ROWS  / 5,000                / 거래 내역 최대 행 수 제한
    * MAX_AMOUNT/ 1조 원                / 거래 한 건당 최대 금액 제한
    * MAX_FILE_BYTES/    5MB           / 업로드 파일 최대 크기 제한

In [2]:
from io import BytesIO      # bytes를 파일처럼 다루는 도구 (업로드 CSV 읽을 때)
from pathlib import Path    # 경로를 Path("폴더") / "파일" 처럼 이어 붙이는 도구

import numpy as np          # 숫자 계산
import pandas as pd         # 표(DataFrame) 다루기

In [3]:
# 상수 : 바뀌지 않는 기준값 (이름은 대문자). 기준이 바뀌면 여기만 고치면 된다
COLUMNS = ['날짜', '구분', '항목', '금액']   # 필수 컬럼 4개
MAX_ROWS = 5000                              # 최대 5,000행
MAX_AMOUNT = 1000000000000                   # 1건당 최대 1조 원
MAX_FILE_BYTES = 5 * 1024 * 1024             # 최대 5MB (컴퓨터는 1024 단위로 셈)

print("필수 컬럼:", COLUMNS)
print("최대 행 수 :", MAX_ROWS)
print("최대 금액:", f"{MAX_AMOUNT:,}원")              # :, → 천 단위 쉼표
print("최대 CSV 크기:", f"{MAX_FILE_BYTES:,}bytes")

필수 컬럼: ['날짜', '구분', '항목', '금액']
최대 행 수 : 5000
최대 금액: 1,000,000,000,000원
최대 CSV 크기: 5,242,880bytes


## 2. 빈 데이터 만들기: empty_records()
* 데이터가 아직 없더라도 열 이름과 자료형을 미리 지정한 빈 DataFrame을 반환

In [4]:
# 행은 0개지만 컬럼과 자료형은 정해진 "빈 표 틀"을 만든다
# → 데이터가 없어도 df['금액'] 같은 코드가 에러 나지 않게
def empty_records() -> pd.DataFrame:     # -> : 반환 자료형 표시(타입 힌트)
    return pd.DataFrame({
        '날짜': pd.Series(dtype="string"),
        '구분': pd.Series(dtype="string"),
        '항목': pd.Series(dtype="string"),
        '금액': pd.Series(dtype="int64"),    # 정수
    })

# 테스트
empty_df = empty_records()
display(empty_df)
print("----")
print(empty_df.dtypes)     # string, string, string, int64 면 정상

,날짜,구분,항목,금액


----
날짜    string
구분    string
항목    string
금액     int64
dtype: object


## 3. 데이터 검증 : validate_records()
* 입출금 데이터가 정해진 규첵에 맞는지 검사하고, 문제가 없으면 데이터를 정리해서 반환하는 함수이다. 
* 검증규칙:
    * 필수 열이 모두 있어야 함
    * 최대 5000행까지 허용
    * 날짜는 실제 날짜인 'YYYY-MM-DD' 형식이며 1900~2100년 범위
    * 구분은 수입 또는 지출 이어야 함
    * 항목은 1~40글자여야함
    * 금액은 1원부터 1조 원까지 양의 정수로 가능
    * 잘못된 행이 있으면 삭제하지 않고 'ValueError'를 발생

In [ ]:
# 데이터가 규칙에 맞는지 검사 → 틀리면 ValueError, 맞으면 정리된 표 반환
def validate_records(frame:pd.DataFrame) -> pd.DataFrame:

    # print(frame.columns)
    # print(frame.columns.duplicated())

    # 1) 중복 컬럼 : 같은 이름이 하나라도 있으면 에러
    if frame.columns.duplicated().any():
        raise ValueError("동일한 컬럼 이름이 중복되었습니다.")

    # 2) 필수 컬럼 : COLUMNS 중 frame에 없는 이름만 모으기 (다 있으면 [])
    missing = [name for name in COLUMNS if name not in frame.columns]
    print(missing)   # ⚠ 디버깅용 → 출력에 [] 가 찍히는 원인
    if missing:      # 빈 리스트면 False
        raise ValueError("필수 컬럼이 없습니다: " + ",".join(missing))   # ['금액'] → "금액"

    # 3) 최대 행 수
    if len(frame) > MAX_ROWS:
        raise ValueError(f"내역은 최대{MAX_ROWS:,}행까지 입력할 수 있습니다.")

    # 4) 0행이면 빈 표 틀 반환
    if frame.empty:
        return empty_records()

    # 5) 필수 열만, 복사본으로, 행 번호는 0부터 새로
    clean = frame[COLUMNS].copy().reset_index(drop=True)

    # 6) 글자 열 앞뒤 공백 제거 (" 월급 " → "월급")
    for name in ["날짜",'구분', '항목']:
        clean[name] = clean[name].astype("string").str.strip()


    # 7) 값 검사 : 행마다 True(정상)/False(오류) 마스크 만들기
    # [날짜] coerce : 변환 못 하면 에러 대신 NaT(빈 날짜)
    date_text = clean["날짜"]
    dates = pd.to_datetime(date_text, format="%Y-%m-%d", errors="coerce")
    valid_dates = (
        date_text.str.fullmatch(r"\d{4}-\d{2}-\d{2}", na=False)  # 0000-00-00 모양
        & dates.notna()                                            # 실제 있는 날짜 (2026-02-30 X)
        & dates.between("1900-01-01","2100-12-31")                 # 허용 범위
    )

    # [구분] 수입/지출만
    valid_kind = clean["구분"].isin(['수입','지출'])
    # [항목] 비어 있지 않고 1~40글자
    valid_category = clean['항목'].notna() & clean["항목"].str.len().between(1,40)
    # [금액] 글자 → 숫자 (못 바꾸면 NaN)
    raw_amount = clean['금액'].astype("string").str.strip()
    amount = pd.to_numeric(raw_amount, errors="coerce")
    valid_amount = (
        amount.notna() & np.isfinite(amount)                    # 숫자이고 무한대 아님
        & amount.between(1, MAX_AMOUNT) & amount.mod(1).eq(0)   # 1원~1조 원, 정수
    ).fillna(False)


    # 8) (설명, 마스크) 쌍을 하나씩 확인 → False가 있으면 에러
    for label, mask in[
        ("날짜: YYYY-MM-DD의 실제 날짜 (1900~2100년)",valid_dates),
        ("구분: 수입 또는 지출",valid_kind),
        ("항목: 1~40글자의 이름", valid_category),
        ("금액: 1원~1조원 양의 정수",valid_amount),
    ]:
        if not mask.all():   # 하나라도 False면
            # ~mask : 뒤집기 → 틀린 행 번호(1부터), 최대 10개
            rows= [str(i + 1) for i in clean. index[~mask ] [ : 10] ]
            raise ValueError(f"{label}를 확인하세요. 잘못된 데이터 행: {', '.join(rows)}")

    # 9) 통과 → 날짜는 "YYYY-MM-DD" 글자로, 금액은 정수로
    clean['날짜'] = dates.dt.strftime("%Y-%m-%d").astype("string")
    clean['금액'] = amount.astype("int64")
    return clean

In [6]:
# test code
# import pandas as pd

# clean = pd.DataFrame({
#     "구분": ["수입", "지출", "저축", "지출", "저축", "수입"]
# })

# mask = clean["구분"].isin(["수입", "지출"])

# print(clean)
# print(mask)
# print()
# print(~mask)

### 3-1. 정상 데이터 검증하기
문자열 앞뒤 공백을 제거하고, 금액을 정수형으로 변환하는지 확인

In [7]:
# 정상 데이터 테스트 : 공백 제거 + 금액 정수 변환 확인
sample = pd.DataFrame({
    "날짜": ["2026-01-05", "2026-01-08", "2026-02-03", "2026-02-10"],
    "구분": ["수입", "지출", "수입", "지출"],
    "항목": [" 월급 ", "식비", "월급", "교통비"],     # 공백 제거 확인용
    "금액": [3_000_000, "12000", 3_000_000, 50000],   # "12000" → 12000 확인용 (_는 읽기용 구분자)
})

clean = validate_records(sample)
display(clean)
print("\n자료형:")
print(clean.dtypes)    # 금액이 int64면 성공

[]


,날짜,구분,항목,금액
0,2026-01-05,수입,월급,3000000
1,2026-01-08,지출,식비,12000
2,2026-02-03,수입,월급,3000000
3,2026-02-10,지출,교통비,50000



자료형:
날짜    string
구분    string
항목    string
금액     int64
dtype: object


In [8]:
# #test code
# 날짜 검사 로직만 따로 떼어 테스트하는 셀 (주석을 풀고 실행하면 결과 확인 가능)
# import pandas as pd

# clean = pd.DataFrame({
#     "날짜": [
#         "2026-01-05",   # 정상 → True
#         "2026-02-30",   # 없는 날짜(2월 30일) → False
#         "2026/03/10",   # 형식이 다름(/ 사용) → False
#         "날짜오류",      # 날짜가 아님 → False
#         "1899-12-31"    # 허용 범위(1900~2100) 밖 → False
#     ]
# })

# date_text = clean["날짜"]

# # 변환할 수 없는 값은 NaT(빈 날짜)가 된다
# dates = pd.to_datetime(
#     date_text,
#     format="%Y-%m-%d",
#     errors="coerce"
# )

# valid_dates = (
#     date_text.str.fullmatch(r"\d{4}-\d{2}-\d{2}", na=False)   # 1) 'YYYY-MM-DD' 모양인가?
#     & dates.notna()                                              # 2) 실제 존재하는 날짜인가?
#     & dates.between("1900-01-01", "2100-12-31")                  # 3) 허용 범위 안인가?
# )

# print(valid_dates)   # 결과: True, False, False, False, False

### 3-2. 잘못된 데이터 시험하기
* 날짜 형식 오류, 잘못된 구분, 0원 금액을 각각 검증

In [9]:
# 잘못된 데이터 테스트 : 에러가 나야 성공
bad_examples = {
    "날짜 오류": pd.DataFrame([["2026/01/05", "수입", "월급", 1000]], columns=COLUMNS),   # / 사용
    "구분 오류": pd.DataFrame([["2026-01-05", "환불", "기타", 1000]], columns=COLUMNS),   # 수입/지출 아님
    "금액 오류": pd.DataFrame([["2026-01-05", "지출", "식비", 0]], columns=COLUMNS),      # 0원
}

for test_name, bad_df in bad_examples.items():   # (키, 값) 하나씩
    try:
        validate_records(bad_df)
    except ValueError as e:                       # 에러를 받아서 출력 (e = raise에 쓴 메시지)
        print(f"[{test_name}] 검증 실패 확인: {e}")
    # 아무것도 안 찍히면 → 그 검사가 제대로 안 된 것

[]
[날짜 오류] 검증 실패 확인: 날짜: YYYY-MM-DD의 실제 날짜 (1900~2100년)를 확인하세요. 잘못된 데이터 행: 1
[]
[구분 오류] 검증 실패 확인: 구분: 수입 또는 지출를 확인하세요. 잘못된 데이터 행: 1
[]
[금액 오류] 검증 실패 확인: 금액: 1원~1조원 양의 정수를 확인하세요. 잘못된 데이터 행: 1


## 4. 샘플 파일 읽기 : load_sample()

In [ ]:
# data 폴더의 샘플 CSV를 읽고 검증까지 해서 반환
def load_sample() -> pd.DataFrame:
    path = Path.cwd() / "data" / "sample_transactions.csv"   # (현재 폴더)/data/파일
    if not path.exists():    # 파일이 없으면 알아보기 쉬운 에러
        raise FileNotFoundError(
            f"샘플 파일을 찾을 수 없습니다. {path}\n"
            "data/sample_transactions.csv파일을 준비해 주세요"
        )
    return validate_records(pd.read_csv(path, encoding="utf-8-sig"))   # utf-8-sig : 엑셀 CSV의 BOM 처리

try:
    loaded = load_sample()
    display(loaded.head())   # 앞 5행
except FileNotFoundError as e:
    print(e)

[]


,날짜,구분,항목,금액
0,2026-01-01,수입,월급,2000000
1,2026-01-05,지출,월세,600000
2,2026-01-10,지출,식비,280000
3,2026-01-15,지출,교통,70000
4,2026-01-20,지출,쇼핑,150000


## 5. CSV 바이트 읽기: read_csv_bytes()
* 파일 업로드 위젯에서 받은 bytes 데이터처럼, 메모리의 CSV 내용을 읽는 함수
* UTF-8 BOM과 CP949 인코딩
* 5MB 초과 파일은 읽지 않음

In [11]:
# 업로드된 CSV(bytes)를 읽어서 검증까지 해서 반환
def read_csv_bytes(content: bytes) -> pd.DataFrame:
    if len(content) > MAX_FILE_BYTES:    # 5MB 초과 거절
        raise ValueError("CSV 파일은 최대 5MB까지 사용할 수 있습니다.")
    if not content:                      # 빈 파일 거절
        raise ValueError("CSV 파일이 비어 있습니다.")

    # utf-8-sig로 읽어 보고, 글자가 깨지면 cp949(옛 한글 엑셀)로 다시 시도
    for encoding in ["utf-8-sig", "cp949"]:
        try:
            # BytesIO : bytes를 파일처럼 / nrows : 5,001행까지만 읽어서 초과 여부만 판단
            frame = pd.read_csv(BytesIO(content), encoding=encoding, nrows=MAX_ROWS + 1)
            return validate_records (frame)
        except UnicodeDecodeError:
            continue     # 다음 인코딩으로
        except (pd.errors.EmptyDataError, pd.errors.ParserError) as exc:
            # 어려운 pandas 에러 → 쉬운 메시지로 바꿔서 다시 발생
            raise ValueError("CSV 형식이 올바르지 않습니다. 쉼표 구분 파일을 사용하세요.") from exc
    raise ValueError("파일 인코딩을 읽을 수 없습니다. UTF-8또는 CP949 CSV로 저장하세요")   # 둘 다 실패

# 테스트 : sample 표 → CSV bytes → 다시 읽기
csv_content = sample.to_csv(index=False).encode("utf-8-sig")
read_back = read_csv_bytes(csv_content)
display(read_back)

[]


,날짜,구분,항목,금액
0,2026-01-05,수입,월급,3000000
1,2026-01-08,지출,식비,12000
2,2026-02-03,수입,월급,3000000
3,2026-02-10,지출,교통비,50000


## 6. 파생 변수 만들기 : add_features()
* 기존 입출금 데이터에 새로운 열인 월과 표시 항목을 추가
    * 월 : 날짜의 앞 7자리로 만든 연-월
    * 표시: 수입이면 '수입', 지출 금액이 기준 이상이면 '큰 지출', 아니면 '일반 지출'
    * 기본 큰 지출 기준은 100,000원 이다.

In [12]:
# 월, 표시 열 추가. threshold(큰 지출 기준)는 생략하면 10만 원
def add_features(frame: pd.DataFrame, threshold: int = 100000) -> pd.DataFrame:
    result = frame.copy()   # 원본 보호

    # 월 : "2026-01-05" → "2026-01" (앞 7글자)
    result["월"] = result["날짜"].apply(lambda d: d[:7])

    # 표시 : 수입 / 큰 지출 / 일반 지출 (axis=1 → 행 하나씩)
    result["표시"] = result.apply(
        lambda row: "수입" if row["구분"] == "수입"
        else ("큰 지출" if row["금액"] >= threshold else "일반 지출"),
        axis=1,
    ) if len(result) else pd.Serise(dtype="string")   # ⚠ 오타 : pd.Series (빈 표일 때만 실행)

    return result


# 테스트
featured = add_features(clean)
display(featured)

,날짜,구분,항목,금액,월,표시
0,2026-01-05,수입,월급,3000000,2026-01,수입
1,2026-01-08,지출,식비,12000,2026-01,일반 지출
2,2026-02-03,수입,월급,3000000,2026-02,수입
3,2026-02-10,지출,교통비,50000,2026-02,일반 지출


## 7. 요약하기: summarize()
* 반환값은 두 개
    1. categories: 항목별 지출 합계, 큰 금액부터 정렬
    2. monthly: 월별 (수입 지출 남은) (금액 남은 비율)

    - 남은 금액 = 수입 - 지출
    - 남은 비율(%) = 남은 금액 ÷ 수입 x 100

In [ ]:
# 항목별 지출 합계, 월별 요약 두 개를 반환
def summarize(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    # 지출만 남기기
    mask = frame["구분"] == "지출"
    expenses = frame[mask]

    # 항목별 지출 합계 → 큰 금액부터
    categories = (
        expenses.groupby("항목")["금액"].sum()
        .sort_values(ascending=False)
        .rename("지출 합계")
        .reset_index()                  # Series → 표
    )

    # 월별 수입/지출 합계 (행=월, 열=구분)
    monthly = frame.pivot_table(
        index="월",
        columns="구분",
        values="금액",
        aggfunc="sum",
        fill_value=0,        # 빈 칸은 0
    )
    # 수입이나 지출 열이 없어도 0으로 만들어 두기
    monthly = monthly.reindex(columns=["수입", "지출"], fill_value=0)

    monthly["남은 금액"] = monthly["수입"] - monthly["지출"]
    # 수입 0원이면 0으로 나누지 않게 NaN으로
    monthly["남은 비율(%)"] = (monthly["남은 금액"] / monthly["수입"].replace(0, np.nan) * 100).round(1)

    monthly = monthly.reset_index()
    return categories, monthly           # 두 개 반환 (튜플)


categories, monthly = summarize(featured)   # 두 개를 각각 받기
print("항목별 지출 합계")
display(categories)
print("월별 요약")
display(monthly)

항목별 지출 합계


,항목,지출 합계
0,교통비,50000
1,식비,12000


월별 요약


구분,월,수입,지출,남은 금액,남은 비율(%)
0,2026-01,3000000,12000,2988000,99.6
1,2026-02,3000000,50000,2950000,98.3


## 8. csv로 내보내기: csv_bytes()
* 엑셀에서 깨지는 현상을 방지하기 위해 utf-8-sig를 사용
* 반환값은 파일 저장이나 streamlit 다운로드에 사용할 수 있는 bytes로 설정

In [ ]:
# 표 → CSV bytes (엑셀 수식 방지 + 한글 깨짐 방지)
def csv_bytes(frame:pd.DataFrame) -> bytes:
    safe = frame.copy()

    # 글자 열에서 = + - @ 등으로 시작하면 앞에 ' 를 붙여 엑셀이 수식으로 실행하지 못하게 (CSV 인젝션 방지)
    for name in safe.select_dtypes(include=['object','string']).columns:
        safe[name] = safe[name].map(
            lambda value: "'" + value
            if isinstance(value,str) and value.lstrip().startswith(("=",'+','-','@','\t','\r'))
            else value
        )
    # index=False : 행 번호 빼기 / na_rep='' : 빈 값은 빈칸
    # utf-8-sig : 엑셀에서 한글이 안 깨지게 (맨 앞에 BOM)
    return safe.to_csv(index= False, na_rep='',encoding=None).encode("utf-8-sig")

export_bytes = csv_bytes(featured)
print("내보낸 데이터 크기 : ", len(export_bytes), "bytes")
print("앞부분 바이트:",export_bytes[:30])   # b'\xef\xbb\xbf = BOM

# 파일로 저장
output_path = Path("transactions_export.csv")
output_path.write_bytes(export_bytes)
print("저장 위치:", output_path.resolve())      # 전체 경로

내보낸 데이터 크기 :  252 bytes
앞부분 바이트: b'\xef\xbb\xbf\xeb\x82\xa0\xec\xa7\x9c,\xea\xb5\xac\xeb\xb6\x84,\xed\x95\xad\xeb\xaa\xa9,\xea\xb8\x88\xec\x95\xa1'


In [ ]:
# test code
'''import pandas as pd

safe = pd.DataFrame({
    '항목' : ['식비', '=1+1', '교통비', '+100'],
    '금액' : [30000,1000,5000,100]
})

# 자료형이 'object','string' 인 것만 처리
for name in safe.select_dtypes(include=['object','string']).columns:
    safe[name] = safe[name].map(
        lambda value: "'" + value
        if isinstance(value,str) and value.lstrip().startswith(("=",'+','-','@','\t','\r'))
        else value
    )
    
safe
'''

,항목,금액
0,식비,30000
1,'=1+1,1000
2,교통비,5000
3,'+100,100


## 9. 종합 : 함수들을 연결하기
* 데이터 -> 검증 -> 파생변수 -> 요약 -> csv 내보내기

In [22]:
# 종합 : 검증 → 열 추가 → 요약 → CSV
raw = pd.DataFrame({
    "날짜": ["2026-03-01", "2026-03-02", "2026-03-05", "2026-04-01"],
    "구분": ["수입", "지출", "지출", "수입"],
    "항목": ["급여", "식비", "월세", "급여"],
    "금액": [3_200_000, 18000, 700000, 3_200_000],
})

validated = validate_records(raw)                            # ① 검증
with_features = add_features(validated, threshold=100000)    # ② 월, 표시 열
by_category, by_month = summarize(with_features)             # ③ 요약 2개
download_content = csv_bytes(with_features)                  # ④ CSV bytes

print("처리된 내역")
display(with_features)
print("항목별 지출")
display(by_category)
print("월별 요약")
display(by_month)
print("CSV 바이트 길이:", len(download_content))

[]
처리된 내역


,날짜,구분,항목,금액,월,표시
0,2026-03-01,수입,급여,3200000,2026-03,수입
1,2026-03-02,지출,식비,18000,2026-03,일반 지출
2,2026-03-05,지출,월세,700000,2026-03,큰 지출
3,2026-04-01,수입,급여,3200000,2026-04,수입


항목별 지출


,항목,지출 합계
0,월세,700000
1,식비,18000


월별 요약


구분,월,수입,지출,남은 금액,남은 비율(%)
0,2026-03,3200000,718000,2482000,77.6
1,2026-04,3200000,0,3200000,100.0


CSV 바이트 길이: 247
